Lesson 1

In [1]:
%load_ext autoreload
%autoreload 2

In [14]:
import os
import dotenv
dotenv.load_dotenv()

True

In [3]:
image_endpoint = os.getenv('IMAGE_ENDPOINT')
image_model = 'Efficient-Large-Model/Sana_1600M_1024px_diffusers'

In [4]:
from gen import ImageGenerator

In [5]:
sana_generator = ImageGenerator(image_endpoint, image_model)

In [6]:
seed_image = sana_generator.generate_image_tool('a desk lamp, a person working from home, dusk')

In [7]:
seed_image

'out\\quickstart\\qwen_20261126-181336_ftm.png'

In [8]:
video_endpoint = os.getenv('VIDEO_ENDPOINT')
video_token = os.getenv('VIDEO_TOKEN')
video_model = 'Wan-AI/Wan2.2-I2V-A14B-Diffusers' 

In [9]:
from gen import VideoGenerator

In [10]:
wan_generator = VideoGenerator(video_endpoint, video_model, token=video_token)

In [11]:
video = wan_generator.generate_video_tool('advertisement', seed_image, 5.0, 16)

In [12]:
video

'out\\quickstart\\wan_20261126-181422_4pu.mp4'

In [18]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

In [ ]:
model = OpenAIChatModel(
    model_name='zai-org/GLM-5.3-Flash',
    provider=OpenAIProvider(
        base_url='https://api.tokenfactory.nebius.com/v1/',
        api_key=os.environ['NEBIUS_API_KEY']
    )
)

In [21]:
from pydantic_ai import Agent

In [29]:
media_instructions = """
You help user create a video advertisement based on the idea they give you. 

First generate an image and then use the image to generate a video. The 
video should be 5 seconds with 16 fps unless user asks for a different 
configuration.
""".strip()

media_agent = Agent(
    model=model,
    tools=[
        sana_generator.generate_image_tool,
        wan_generator.generate_video_tool
    ],
    instructions=media_instructions
)


In [31]:
result = await media_agent.run('a desk lamp, a person working from home, dusk')

In [33]:
print(result.output)

Your ad is ready! Here's what I created:

**The scene:** A warm, cinematic dusk shot — a sleek desk lamp casting amber light over a person working from home on their laptop, with deep blue twilight glowing through the window behind them.

**The video (5s @ 16fps):**
- A slow cinematic push-in toward the glowing lamp
- The person's hands typing gently on the laptop
- Subtle light flicker from the lamp with dust particles drifting through the beam
- Dusk deepening outside the window for that calm, focused evening-work mood

**Files:**
- 🖼️ Image: `out\quickstart\qwen_20261126-184233_b1k.png`
- 🎬 Video: `out\quickstart\wan_20261126-184315_s4z.mp4`

Want a different vibe (brighter, more minimalist, a specific lamp style) or a different camera movement? I can regenerate either the image or the video.


In [34]:
from pydantic_ai.messages import FunctionToolCallEvent

class NamedCallback:

    def __init__(self, agent):
        self.agent_name = agent.name

    async def print_function_calls(self, ctx, event):
        # Detect nested streams
        if hasattr(event, "__aiter__"):
            async for sub in event:
                await self.print_function_calls(ctx, sub)
            return

        if isinstance(event, FunctionToolCallEvent):
            tool_name = event.part.tool_name
            args = event.part.args
            print(f"TOOL CALL ({self.agent_name}): {tool_name}({args})")

    async def __call__(self, ctx, event):
        return await self.print_function_calls(ctx, event)

In [35]:
callback = NamedCallback(media_agent) 

In [36]:
result = await media_agent.run(
    'a desk lamp, a person working from home, dusk',
    event_stream_handler=callback
)

TOOL CALL (media_agent): generate_image_tool({"prompt": "Cinematic advertisement still: a modern minimalist desk lamp glowing warm amber as the hero of the scene, on a tidy wooden desk where a person works from home at dusk. Soft blue-purple twilight light through a large window behind them, laptop open, coffee mug steaming, notebook and plant on desk. Shallow depth of field, warm-cool color contrast, moody product photography, photorealistic, 35mm lens, cozy atmosphere"})
TOOL CALL (media_agent): generate_video_tool({"duration": 5, "fps": 16, "image_path": "out\\quickstart\\qwen_20261126-184847_ezg.png", "prompt": "Slow cinematic camera push-in toward the glowing desk lamp. The person types gently on the laptop, steam rises softly from the coffee mug, warm lamp light subtly pulses, dusk light outside the window slowly deepens. Calm, cozy, premium advertisement mood"})


In [ ]:
print(result.output)

We want the agent to look at the image before generating video

In [40]:
from pathlib import Path
from pydantic_ai import BinaryContent, ToolReturn

def view_image(path: str) -> ToolReturn:
    """
    Look at an image. Use it to check a generated image before making a video from it.
    """
    return ToolReturn(
        return_value=f"Image {path} is attached",
        content=[BinaryContent(data=Path(path).read_bytes(), media_type="image/png")],
    )


In [41]:
media_instructions = """
You help user create a video advertisement based on the idea they give you. 

First generate an image and then use the image to generate a video. The 
video should be 5 seconds with 16 fps unless user asks for a different 
configuration.

Before generating the video, make sure that the image contains what you expect.
If it doesn't, regenerate it with a different prompt.
""".strip()

media_agent = Agent(
    model=model,
    tools=[
        sana_generator.generate_image_tool,
        wan_generator.generate_video_tool,
        view_image,
    ],
    instructions=media_instructions
)


In [42]:
result = await media_agent.run(
    'a desk lamp, a person working from home, dusk',
    event_stream_handler=callback
)

TOOL CALL (media_agent): generate_image_tool({"prompt": "A cozy home office at dusk, warm light from a stylish modern desk lamp glowing softly on a wooden desk, a person seen from behind working on a laptop, large window showing deep blue twilight sky with city lights beginning to twinkle, cinematic warm and moody lighting, shallow depth of field, photorealistic, advertising photography style"})
TOOL CALL (media_agent): view_image({"path": "out\\quickstart\\qwen_20261126-190433_dyr.png"})
TOOL CALL (media_agent): generate_video_tool({"duration": 5, "fps": 16, "image_path": "out\\quickstart\\qwen_20261126-190433_dyr.png", "prompt": "Slow cinematic dolly-in toward a person working at a home office desk at dusk. The desk lamps glow warmly, the person gently types on the laptop, city lights twinkle and clouds drift slowly outside the window, subtle warm light flicker, calm and cozy atmosphere"})
